# 🍕 Food Image Classification (Food-101): CNN vs EfficientNet vs CLIP

**ML project:** recognise the type of dish from a photo. The best model is then used by the
**Klean Nham food search app** (`food_search.ipynb`): upload a photo → the app names the dish, shows how to cook it
(`data/recipes.csv`) and lists places in Phnom Penh that serve it.

| # | Model | Idea | Training |
|---|---|---|---|
| 1 | **Small CNN** (from scratch) | baseline: what a network learns with no prior knowledge | all layers, random start |
| 2 | **EfficientNet-B0** (transfer learning) | reuse features learned on ImageNet (1.28M photos) | new head first, then fine-tune everything |
| 3 | **CLIP ViT-B/32** (zero-shot) | match the photo to the *text* "a photo of pizza" | **no training at all** |

**Dataset:** [Food-101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/) (Bossard et al., 2014; on Kaggle as
*kmader/food41*): 101 dishes × 1,000 photos = 101,000 photos. Official split: 750 train + 250 test photos per dish.
The *training* photos were deliberately left a little noisy (some wrong labels, odd colours); the *test* photos were
checked by hand.

**Where to run it**

| Where | How | Time (full run) |
|---|---|---|
| **Kaggle** (recommended) | New notebook → *File → Import notebook* → this file. *Add Input* → search **food41** (kmader). Settings: **Accelerator GPU T4**, **Internet on**. *Run all*. | ~1.5–2 h |
| **Google Colab** | Upload this notebook, *Runtime → Change runtime type → T4 GPU*, *Run all* (downloads Food-101, 5 GB) | ~2 h |
| **This Mac** (M3 Pro) | Kernel *venv (food-search)*. Needs ~10 GB free disk for the download. Without an NVIDIA GPU the notebook switches to a **quick run** (fewer photos and epochs). | ~1–2 h quick run |

**What you get:** `models/food101_efficientnet_b0.pt` + `models/food101_classes.json` (used by the app) and
`results/food_classification/` (tables and figures for the report). On Kaggle/Colab, download
`food_classification_outputs.zip` at the end and unzip it into the project folder.

## 0. Setup

In [ ]:
# Where are we running? Kaggle and Colab give a free NVIDIA GPU; a Mac works for a quick run.
import os, sys, json, time, random, copy, shutil, platform, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").exists()
if IN_KAGGLE:
    ROOT = Path("/kaggle/working")
elif IN_COLAB:
    ROOT = Path("/content")
else:
    ROOT = Path.cwd()          # VS Code starts the kernel in the notebook's folder (the project root)
print("Kaggle:", IN_KAGGLE, "| Colab:", IN_COLAB, "| root:", ROOT)

if IN_COLAB or IN_KAGGLE:      # torch, torchvision, sklearn are preinstalled there; CLIP is not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split


def pick_device() -> str:
    # Prefer an NVIDIA GPU, then Apple Silicon (MPS), then CPU.
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device()
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", DEVICE,
      f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")

MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results" / "food_classification"
for folder in (MODELS_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Chart style (same as food_search.ipynb): recessive grid/axes, fixed colour per model
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.8, "axes.axisbelow": True,
                     "axes.edgecolor": "#b5b4ae", "axes.labelcolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e", "figure.dpi": 110})

### Settings

A **full run** uses all 101,000 photos and needs an NVIDIA GPU (Kaggle/Colab). Without one, `QUICK_RUN` is switched on:
150 training photos and 50 test photos per dish, fewer epochs. The numbers are lower, but every step and figure is
the same. Write in the report which one you ran.

In [ ]:
QUICK_RUN = DEVICE != "cuda"      # set to False to force the full run (slow without an NVIDIA GPU)
SMOKE_TEST = os.environ.get("FOOD_SMOKE_TEST") == "1"   # tiny run that only checks the code works

CONFIG = dict(
    val_fraction=0.10,            # 10% of the training photos are kept aside to pick the best epoch
    train_per_class=None,         # None = all 750 photos per dish
    test_per_class=None,          # None = all 250
    batch_size=64,
    num_workers=0 if platform.system() == "Darwin" else min(4, os.cpu_count() or 1),
    label_smoothing=0.1, weight_decay=1e-4,
    cnn_img=128, cnn_epochs=15, cnn_lr=3e-3,                       # model 1
    eff_img=224, eff_head_epochs=2, eff_head_lr=1e-3,              # model 2, step 1: new head only
    eff_finetune_epochs=6, eff_lr=3e-4,                            # model 2, step 2: all layers
)
if QUICK_RUN:
    CONFIG.update(train_per_class=150, test_per_class=50, cnn_epochs=8, eff_head_epochs=1, eff_finetune_epochs=3)
if SMOKE_TEST:
    CONFIG.update(train_per_class=10, test_per_class=3, batch_size=8, cnn_img=64, eff_img=64,
                  cnn_epochs=1, eff_head_epochs=1, eff_finetune_epochs=1)
print("Quick run:" if QUICK_RUN else "Full run:", json.dumps(CONFIG, indent=1))

## 1. Data

The notebook looks for Food-101 in this order: `FOOD101_DIR` (environment variable), Kaggle input (`food41`),
`data/food101/`. If it is not found, it downloads the official archive (5 GB) with `torchvision`.

In [ ]:
def find_food101(bases):
    # Return (images_dir, meta_dir) of a Food-101 copy: meta/ has classes.txt, train.txt, test.txt.
    for base in bases:
        base = Path(base)
        if not base.exists():
            continue
        for classes_txt in base.rglob("classes.txt"):
            meta = classes_txt.parent
            if not ((meta / "train.txt").exists() and (meta / "test.txt").exists()):
                continue
            for up in (meta.parent, meta.parent.parent):      # official: food-101/meta; Kaggle food41: meta/meta
                if (up / "images").is_dir():
                    return up / "images", meta
    return None, None


DOWNLOAD_DIR = Path(os.environ.get("FOOD101_DIR") or ROOT / "data" / "food101")
IMAGES_DIR, META_DIR = find_food101([DOWNLOAD_DIR, "/kaggle/input"])

if IMAGES_DIR is None:
    free_gb = shutil.disk_usage(ROOT).free / 1e9
    print(f"Food-101 not found: downloading to {DOWNLOAD_DIR} (5 GB download, 10 GB while unpacking; {free_gb:.0f} GB free)")
    if free_gb < 11:
        raise SystemExit("Not enough free disk space. Free up space, or run this notebook on Kaggle/Colab.")
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    torchvision.datasets.Food101(root=DOWNLOAD_DIR, split="train", download=True)
    (DOWNLOAD_DIR / "food-101.tar.gz").unlink(missing_ok=True)        # keep only the unpacked photos
    IMAGES_DIR, META_DIR = find_food101([DOWNLOAD_DIR])

print("images:", IMAGES_DIR)
print("meta:  ", META_DIR)

In [ ]:
CLASSES = (META_DIR / "classes.txt").read_text().split()
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

NICE_NAMES = {"caesar_salad": "Caesar salad", "creme_brulee": "Crème brûlée", "eggs_benedict": "Eggs Benedict",
              "french_fries": "French fries", "french_onion_soup": "French onion soup", "french_toast": "French toast",
              "greek_salad": "Greek salad", "pho": "Pho", "beef_carpaccio": "Beef carpaccio"}


def display_name(c: str) -> str:
    return NICE_NAMES.get(c, c.replace("_", " ").capitalize())


def read_split(name: str) -> pd.DataFrame:
    # Lines look like "apple_pie/1005649" -> images/apple_pie/1005649.jpg
    lines = (META_DIR / f"{name}.txt").read_text().split()
    return pd.DataFrame({"path": [str(IMAGES_DIR / f"{l}.jpg") for l in lines],
                         "label": [CLASS_TO_IDX[l.split("/")[0]] for l in lines]})


def per_class(frame, n):
    if not n:
        return frame.reset_index(drop=True)
    return (frame.sample(frac=1, random_state=SEED).groupby("label", group_keys=False).head(n)
            .sort_values("label").reset_index(drop=True))


train_full = per_class(read_split("train"), CONFIG["train_per_class"])
test_df = per_class(read_split("test"), CONFIG["test_per_class"])
train_df, val_df = train_test_split(train_full, test_size=CONFIG["val_fraction"], stratify=train_full["label"],
                                    random_state=SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

print(f"{NUM_CLASSES} dishes | train {len(train_df):,} | validation {len(val_df):,} | test {len(test_df):,}")
print("photos per dish (train):", train_df.label.value_counts().agg(["min", "max"]).to_dict())
print("first dishes:", ", ".join(display_name(c) for c in CLASSES[:8]), "...")

In [ ]:
# A look at the data: one random training photo for 18 dishes
picks = train_df.groupby("label").sample(1, random_state=SEED).sample(min(18, NUM_CLASSES), random_state=SEED)
fig, axes = plt.subplots(3, 6, figsize=(13, 7))
for ax in axes.flat:
    ax.axis("off")
for ax, (_, r) in zip(axes.flat, picks.iterrows()):
    img = Image.open(r.path).convert("RGB")
    ax.imshow(img); ax.set_title(f"{display_name(CLASSES[r.label])}\n{img.size[0]}×{img.size[1]}", fontsize=9)
fig.suptitle("Food-101 training photos (title: dish, size in pixels)")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "sample_photos.png", bbox_inches="tight"); plt.show()

### From photo to tensor

* **Training:** random crop (35–100% of the photo), random flip, and *TrivialAugment* (one random colour/shape change
  per photo). The network sees a slightly different version of each photo every epoch, which reduces overfitting.
* **Validation / test:** always the same: resize, then a centre crop.
* Pixel values are normalised with the ImageNet mean and standard deviation (what EfficientNet was trained with).

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def train_transform(size):
    return T.Compose([T.RandomResizedCrop(size, scale=(0.35, 1.0)), T.RandomHorizontalFlip(),
                      T.TrivialAugmentWide(), T.ToTensor(), T.Normalize(MEAN, STD)])


def eval_transform(size):
    return T.Compose([T.Resize(int(size * 1.14)), T.CenterCrop(size), T.ToTensor(), T.Normalize(MEAN, STD)])


class FoodDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths, self.labels, self.transform = frame["path"].tolist(), frame["label"].tolist(), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB")), self.labels[i]


def make_loader(frame, transform, shuffle=False):
    workers = CONFIG["num_workers"]
    return DataLoader(FoodDataset(frame, transform), batch_size=CONFIG["batch_size"], shuffle=shuffle,
                      num_workers=workers, pin_memory=DEVICE == "cuda", persistent_workers=workers > 0)


x, y = next(iter(make_loader(train_df, train_transform(CONFIG["eff_img"]), shuffle=True)))
print("one batch:", tuple(x.shape), "labels:", y[:8].tolist())

## 2. Training loop (shared by models 1 and 2)

* **Loss:** cross-entropy with *label smoothing* 0.1 (the target is 90% the right dish instead of 100%), which helps
  with Food-101's noisy training labels.
* **Optimiser:** AdamW. **Learning rate:** *one-cycle* schedule (warm up, then slowly down to almost 0).
* After every epoch we measure the **validation** accuracy and keep the weights of the best epoch.
  The **test** set is used only once, at the end (section 6).
* **Top-1 accuracy:** the first guess is right. **Top-5:** the right dish is among the 5 best guesses.

In [ ]:
USE_AMP = DEVICE == "cuda"        # mixed precision (float16) on NVIDIA GPUs: about 2× faster
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
HISTORY, TRAIN_SECONDS = [], {}


def autocast():
    return torch.autocast(device_type="cuda" if USE_AMP else "cpu", dtype=torch.float16, enabled=USE_AMP)


def run_epoch(model, loader, optimizer=None, scheduler=None, scaler=None):
    # One pass over the data. Trains when an optimizer is given. Returns (loss, top-1, top-5).
    training = optimizer is not None
    model.train(training)
    n_seen, loss_sum, right1, right5 = 0, 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            top5 = logits.topk(min(5, logits.size(1)), dim=1).indices
            n_seen += y.size(0)
            loss_sum += loss.item() * y.size(0)
            right1 += (top5[:, 0] == y).sum().item()
            right5 += (top5 == y[:, None]).any(dim=1).sum().item()
    return loss_sum / n_seen, right1 / n_seen, right5 / n_seen


def fit(model, name, epochs, lr, train_loader, val_loader, stage=""):
    # Train the parameters that require gradients; keep (and finally load) the best epoch on validation.
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * len(train_loader),
                                                    pct_start=0.15)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    past = [h for h in HISTORY if h["model"] == name]
    best = max((h["val_top1"] for h in past), default=-1.0)
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()} if past else None
    for _ in range(epochs):
        start = time.time()
        tr = run_epoch(model, train_loader, optimizer, scheduler, scaler)
        va = run_epoch(model, val_loader)
        seconds = time.time() - start
        TRAIN_SECONDS[name] = TRAIN_SECONDS.get(name, 0) + seconds
        row = dict(model=name, stage=stage, epoch=len(past) + 1, train_loss=tr[0], train_top1=tr[1],
                   val_loss=va[0], val_top1=va[1], val_top5=va[2], seconds=round(seconds))
        HISTORY.append(row); past.append(row)
        marker = ""
        if va[1] > best:
            best, marker = va[1], "  ★ best so far"
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"{name} {stage} epoch {row['epoch']:>2}: train loss {tr[0]:.3f} acc {tr[1]:.1%} | "
              f"val loss {va[0]:.3f} top-1 {va[1]:.1%} top-5 {va[2]:.1%} | {seconds:.0f}s{marker}")
    model.load_state_dict(best_state)
    return model


def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters())

## 3. Model 1: a small CNN trained from scratch (baseline)

Five blocks of *conv 3×3 → batch norm → ReLU* (twice) and *max-pool*; the image shrinks from 128×128 to 4×4 while
the number of channels grows from 3 to 384. Then *global average pooling*, dropout and one linear layer that gives a
score for each of the 101 dishes. It starts from random weights, so everything it knows comes from Food-101.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.MaxPool2d(2))


class SmallCNN(nn.Module):
    def __init__(self, num_classes, widths=(32, 64, 128, 256, 384)):
        super().__init__()
        blocks, c_in = [], 3
        for w in widths:
            blocks.append(conv_block(c_in, w))
            c_in = w
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(c_in, num_classes))

    def forward(self, x):
        return self.head(self.features(x))


cnn = SmallCNN(NUM_CLASSES).to(DEVICE)
print(f"Small CNN: {count_params(cnn):,} parameters")

In [ ]:
size = CONFIG["cnn_img"]
cnn = fit(cnn, "Small CNN", CONFIG["cnn_epochs"], CONFIG["cnn_lr"],
          make_loader(train_df, train_transform(size), shuffle=True), make_loader(val_df, eval_transform(size)))
torch.save({"arch": "small_cnn", "state_dict": cnn.state_dict(), "classes": CLASSES, "img_size": size},
           MODELS_DIR / "food101_small_cnn.pt")

## 4. Model 2: EfficientNet-B0 with transfer learning

EfficientNet-B0 (Tan & Le, 2019) was trained on ImageNet: it already knows edges, textures and shapes. We replace its
last layer (1,000 ImageNet classes) with a new one for our 101 dishes and train in two steps:

1. **Head only** (`eff_head_epochs`): the pretrained layers are frozen; only the new layer learns. This way the random
   new layer cannot damage the good pretrained features.
2. **Fine-tuning** (`eff_finetune_epochs`): all layers learn, with a smaller learning rate.

In [ ]:
def build_efficientnet(num_classes, pretrained=True):
    weights = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    model = torchvision.models.efficientnet_b0(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model


eff = build_efficientnet(NUM_CLASSES).to(DEVICE)
size = CONFIG["eff_img"]
eff_train_loader = make_loader(train_df, train_transform(size), shuffle=True)
eff_val_loader = make_loader(val_df, eval_transform(size))

# Step 1: head only
for p in eff.features.parameters():
    p.requires_grad = False
trainable = sum(p.numel() for p in eff.parameters() if p.requires_grad)
print(f"EfficientNet-B0: {count_params(eff):,} parameters, {trainable:,} trainable in step 1")
eff = fit(eff, "EfficientNet-B0", CONFIG["eff_head_epochs"], CONFIG["eff_head_lr"],
          eff_train_loader, eff_val_loader, stage="head")

In [ ]:
# Step 2: fine-tune all layers
for p in eff.parameters():
    p.requires_grad = True
eff = fit(eff, "EfficientNet-B0", CONFIG["eff_finetune_epochs"], CONFIG["eff_lr"],
          eff_train_loader, eff_val_loader, stage="fine-tune")

## 5. Model 3: CLIP zero-shot (no training)

CLIP (Radford et al., 2021) learned from 2 billion photo + caption pairs to put a photo and its description close
together in the same vector space. We use the same CLIP the search app already uses (ViT-B/32, LAION-2B weights).

**Zero-shot:** for each dish we write sentences like *"a photo of pizza, a type of food."* and turn them into vectors.
A test photo gets the dish whose sentence vector is closest (cosine similarity). It has **never seen a Food-101
training photo**.

In [ ]:
import open_clip

CLIP_ARCH, CLIP_PRETRAINED = "ViT-B-32", "laion2b_s34b_b79k"
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_PRETRAINED,
                                                                       device=DEVICE)
clip_model.eval()
clip_tokenizer = open_clip.get_tokenizer(CLIP_ARCH)

TEMPLATES = ["a photo of {}, a type of food.", "a close-up photo of {}.", "a photo of a plate of {}.",
             "{}, food photography."]


@torch.no_grad()
def dish_text_vectors(names):
    # One vector per dish: the average of its template sentences (prompt ensembling).
    vectors = []
    for name in names:
        tokens = clip_tokenizer([t.format(name.lower()) for t in TEMPLATES]).to(DEVICE)
        v = clip_model.encode_text(tokens).float()
        v = (v / v.norm(dim=-1, keepdim=True)).mean(dim=0)
        vectors.append(v / v.norm())
    return torch.stack(vectors)


class ClipZeroShot(nn.Module):
    # Photo -> similarity to every dish sentence, as logits (×100, CLIP's temperature).
    def __init__(self, model, text_vectors):
        super().__init__()
        self.model, self.text = model, text_vectors

    def forward(self, x):
        img = self.model.encode_image(x).float()
        return 100 * (img / img.norm(dim=-1, keepdim=True)) @ self.text.T


clip_zero = ClipZeroShot(clip_model, dish_text_vectors([display_name(c) for c in CLASSES]))
print(f"CLIP image encoder: {count_params(clip_model.visual):,} parameters | dish vectors: {tuple(clip_zero.text.shape)}")

## 6. Test-set evaluation and comparison

All three models are scored on the same **test** photos, which none of them saw during training or model selection.

* **Top-1 / top-5 accuracy** (see section 2).
* **Macro F1:** F1 per dish (balance of precision and recall), averaged so every dish counts the same.
* **ms per photo:** prediction time on this machine, batch of 64.

In [ ]:
@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    out, start = [], time.time()
    for x, _ in loader:
        with autocast():
            out.append(model(x.to(DEVICE)).float().softmax(dim=1).cpu())
    return torch.cat(out).numpy(), (time.time() - start) / len(loader.dataset) * 1000


MODELS = {"Small CNN": (cnn, eval_transform(CONFIG["cnn_img"]), count_params(cnn)),
          "EfficientNet-B0": (eff, eval_transform(CONFIG["eff_img"]), count_params(eff)),
          "CLIP zero-shot": (clip_zero, clip_preprocess, count_params(clip_model.visual))}
y_true = test_df["label"].to_numpy()
PROBS, rows = {}, []
for name, (model, tf, n_params) in MODELS.items():
    probs, ms = predict_probs(model, make_loader(test_df, tf))
    PROBS[name] = probs
    top5 = np.argsort(-probs, axis=1)[:, :5]
    rows.append({"model": name, "top1_acc": (top5[:, 0] == y_true).mean(), "top5_acc": (top5 == y_true[:, None]).any(1).mean(),
                 "macro_f1": f1_score(y_true, top5[:, 0], average="macro"), "params_M": n_params / 1e6,
                 "ms_per_photo": ms, "train_minutes": TRAIN_SECONDS.get(name, 0) / 60})
    print(f"{name:16s} top-1 {rows[-1]['top1_acc']:.1%}  top-5 {rows[-1]['top5_acc']:.1%}")

metrics = pd.DataFrame(rows).set_index("model")
metrics.to_csv(RESULTS_DIR / "metrics.csv")
pd.DataFrame(HISTORY).to_csv(RESULTS_DIR / "training_history.csv", index=False)
metrics.style.format({"top1_acc": "{:.1%}", "top5_acc": "{:.1%}", "macro_f1": "{:.3f}", "params_M": "{:.1f}",
                      "ms_per_photo": "{:.1f}", "train_minutes": "{:.0f}"})

In [ ]:
# Accuracy per model: top-1 and top-5 side by side, values written on the bars
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col, title in zip(axes, ["top1_acc", "top5_acc"], ["Top-1 accuracy (test)", "Top-5 accuracy (test)"]):
    values = metrics[col].to_numpy() * 100
    bars = ax.barh(metrics.index, values, color=SERIES[:len(values)], height=0.55)
    for bar, v in zip(bars, values):
        ax.text(v + 1, bar.get_y() + bar.get_height() / 2, f"{v:.1f}%", va="center", color="#2b2a27", fontsize=10)
    ax.set_xlim(0, 105); ax.set_title(title, loc="left"); ax.grid(axis="y", visible=False)
axes[0].invert_yaxis()
plt.tight_layout(); plt.savefig(RESULTS_DIR / "accuracy_comparison.png", bbox_inches="tight"); plt.show()

In [ ]:
# Learning curves: validation top-1 per epoch, and train vs validation loss (overfitting check)
hist = pd.DataFrame(HISTORY)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for color, (name, h) in zip(SERIES, hist.groupby("model", sort=False)):
    axes[0].plot(h.epoch, h.val_top1 * 100, marker="o", markersize=5, lw=2, color=color, label=name)
    axes[1].plot(h.epoch, h.train_loss, lw=2, color=color, label=f"{name} train")
    axes[1].plot(h.epoch, h.val_loss, lw=2, ls="--", color=color, label=f"{name} validation")
head_end = hist[(hist.model == "EfficientNet-B0") & (hist.stage == "head")].epoch.max()
if pd.notna(head_end):
    axes[0].axvline(head_end + 0.5, color="#b5b4ae", lw=1)
    axes[0].annotate("fine-tuning starts", (head_end + 0.6, axes[0].get_ylim()[0] + 2), color="#52514e", fontsize=9)
from matplotlib.ticker import MaxNLocator
for ax, title in zip(axes, ["Validation top-1 accuracy (%)", "Loss: train (solid) vs validation (dashed)"]):
    ax.set_title(title, loc="left"); ax.set_xlabel("epoch"); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
axes[0].legend(frameon=False); axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "learning_curves.png", bbox_inches="tight"); plt.show()

### Which dishes get confused? (best model)

The confusion matrix has one row per true dish and one column per predicted dish; a perfect model only has the
diagonal. With 101 dishes the full matrix is large, so the table below lists the pairs that are mixed up most often.

In [ ]:
BEST = metrics["top1_acc"].idxmax()
pred_best = PROBS[BEST].argmax(1)
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
np.add.at(cm, (y_true, pred_best), 1)
cm_rate = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(12, 11))
im = ax.imshow(cm_rate, cmap="Blues", vmin=0, vmax=1)
ticks = np.arange(NUM_CLASSES)
ax.set_xticks(ticks, [display_name(c) for c in CLASSES], rotation=90, fontsize=4.5)
ax.set_yticks(ticks, [display_name(c) for c in CLASSES], fontsize=4.5)
ax.grid(False); ax.set_xlabel("predicted dish"); ax.set_ylabel("true dish")
ax.set_title(f"Confusion matrix, {BEST} (share of each true dish)", loc="left")
fig.colorbar(im, fraction=0.03, pad=0.01)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "confusion_matrix.png", bbox_inches="tight", dpi=200); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
pairs = [(off[i, j], i, j) for i, j in zip(*np.nonzero(off))]
confused = pd.DataFrame([{"true dish": display_name(CLASSES[i]), "predicted as": display_name(CLASSES[j]),
                          "photos": n, "share of true dish": f"{cm_rate[i, j]:.0%}"}
                         for n, i, j in sorted(pairs, reverse=True)[:15]])
confused.to_csv(RESULTS_DIR / "most_confused_pairs.csv", index=False)
confused

In [ ]:
# Per-dish accuracy for all three models; easiest and hardest dishes for the best model
per_dish = pd.DataFrame({name: [(p.argmax(1)[y_true == c] == c).mean() if (y_true == c).any() else np.nan
                                for c in range(NUM_CLASSES)] for name, p in PROBS.items()},
                        index=[display_name(c) for c in CLASSES])
per_dish.to_csv(RESULTS_DIR / "per_dish_accuracy.csv")

ranked = per_dish[BEST].sort_values()
show = pd.concat([ranked.head(10), ranked.tail(10)]) if len(ranked) > 20 else ranked
fig, ax = plt.subplots(figsize=(8, 6.5))
bars = ax.barh(show.index, show.to_numpy() * 100, color=SERIES[list(MODELS).index(BEST)], height=0.6)
for bar, v in zip(bars, show.to_numpy() * 100):
    ax.text(v + 1, bar.get_y() + bar.get_height() / 2, f"{v:.0f}%", va="center", fontsize=8, color="#2b2a27")
if len(ranked) > 20:
    ax.axhline(9.5, color="#b5b4ae", lw=1)
ax.set_xlim(0, 110); ax.grid(axis="y", visible=False)
ax.set_title(f"{BEST}: 10 hardest (bottom) and 10 easiest (top) dishes, test top-1 %", loc="left")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "hardest_easiest_dishes.png", bbox_inches="tight"); plt.show()

In [ ]:
# Error analysis: the best model's most confident mistakes
conf = PROBS[BEST].max(1)
wrong = np.where(pred_best != y_true)[0]
worst = wrong[np.argsort(-conf[wrong])][:12]
fig, axes = plt.subplots(2, 6, figsize=(14, 5.5))
for ax in axes.flat:
    ax.axis("off")
for ax, i in zip(axes.flat, worst):
    ax.imshow(Image.open(test_df.path[i]).convert("RGB"))
    ax.set_title(f"true: {display_name(CLASSES[y_true[i]])}\npred: {display_name(CLASSES[pred_best[i]])} ({conf[i]:.0%})",
                 fontsize=8)
fig.suptitle(f"{BEST}: most confident mistakes", x=0.01, ha="left")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "confident_mistakes.png", bbox_inches="tight"); plt.show()

## 7. Try it on any photo

`classify_photo(path)` returns the 5 most likely dishes with their probability. This is exactly what the app does
when someone uploads a photo.

In [ ]:
APP_TRANSFORM = eval_transform(CONFIG["eff_img"])


@torch.no_grad()
def classify_photo(path, k=5, model=None):
    model = (model or eff).eval()
    x = APP_TRANSFORM(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)
    probs = model(x).float().softmax(dim=1)[0].cpu()
    p, idx = probs.topk(min(k, NUM_CLASSES))
    return [(display_name(CLASSES[i]), round(float(q), 3)) for q, i in zip(p, idx)]


sample = test_df.sample(1, random_state=7).iloc[0]
print("true dish:", display_name(CLASSES[sample.label]))
classify_photo(sample.path)

## 8. Export the model for the app

The search app (`backend/food_search/classify.py`, generated by `food_search.ipynb`) loads these two files:

* `models/food101_efficientnet_b0.pt`: the weights + the settings needed to prepare a photo the same way.
* `models/food101_classes.json`: the 101 dish ids and display names (the ids match `data/recipes.csv`).

In [ ]:
best_eff = metrics.loc["EfficientNet-B0"]
torch.save({"arch": "efficientnet_b0", "state_dict": {k: v.cpu() for k, v in eff.state_dict().items()},
            "classes": CLASSES, "img_size": CONFIG["eff_img"], "resize": int(CONFIG["eff_img"] * 1.14),
            "mean": MEAN, "std": STD, "quick_run": QUICK_RUN,
            "test_top1": float(best_eff.top1_acc), "test_top5": float(best_eff.top5_acc),
            "trained": time.strftime("%Y-%m-%d")}, MODELS_DIR / "food101_efficientnet_b0.pt")
(MODELS_DIR / "food101_classes.json").write_text(
    json.dumps([{"id": c, "name": display_name(c)} for c in CLASSES], indent=1, ensure_ascii=False), encoding="utf-8")
print("saved:", *(f"  {p.relative_to(ROOT)}  {p.stat().st_size / 1e6:.1f} MB" for p in sorted(MODELS_DIR.glob("food101_*"))), sep="\n")

if IN_KAGGLE or IN_COLAB:   # one zip to download and unzip into the project folder
    archive = shutil.make_archive(str(ROOT / "food_classification_outputs"), "zip", ROOT, "models")
    shutil.make_archive(str(ROOT / "_results"), "zip", ROOT, "results")
    import zipfile
    with zipfile.ZipFile(archive, "a") as z, zipfile.ZipFile(ROOT / "_results.zip") as r:
        for item in r.namelist():
            z.writestr(item, r.read(item))
    (ROOT / "_results.zip").unlink()
    print("download:", archive)

## 9. Summary (fill in after your run)

| Model | Top-1 | Top-5 | Trained on Food-101? |
|---|---|---|---|
| Small CNN | see `metrics.csv` | | yes, from scratch |
| EfficientNet-B0 | | | yes, transfer learning |
| CLIP zero-shot | | | no |

Points to discuss in the report:

* **Transfer learning vs from scratch:** how much do ImageNet features help, with the same data and fewer epochs?
* **Zero-shot vs fine-tuned:** CLIP needs no labelled photos at all. When is that good enough, and what does
  fine-tuning add?
* **Errors:** which dishes get confused (`most_confused_pairs.csv`), and why (similar look, noisy labels, mixed plates)?
* **Overfitting:** the gap between train and validation loss in the learning curves.
* **Limits:** Food-101 has no Cambodian dishes; a photo of amok gets the nearest Food-101 dish. The app shows
  the confidence and asks the user to confirm when it is low.